# V-JEPA-Geo v46 — CO3D, Kaggle T4 ×2

        Corrected standalone notebook. Select **GPU T4 ×2**, turn **Internet on**, and run the cells in order.
        Release: **2026-11-03 — experimental relative-geometry and calibration revision**.
        **No dataset attachments required.** CO3D and TUM download missing official data into `/kaggle/temp`.
        The pretrained encoder also downloads into `/kaggle/temp`; checkpoints, figures and tables save to `/kaggle/working`.
        The embedded code is also supplied as readable `kaggle_v46/*.py` in the repository.

        This starts a **new corrected experiment**. V45 improved camera ATE but regressed in depth and calibration. V46 changes training objectives and calibration gradients; benchmark gains are unmeasured. V44/v45 weights cannot resume this architecture.
        The full main schedule is **20 epochs = 5 frozen + 10 partial-backbone + 5 alignment/refinement**.
        On two GPUs, microbatch 1 and accumulation 2 give global batch 4. Evaluation uses five views from every ten-view window.
        All test metrics and figure exports come from the globally best validation depth/pose composite checkpoint.
        A session limit saves an incomplete checkpoint; it does not shorten the experiment or mark it complete.

V46 retains spatial matching and adds dedicated calibration context with isolated
        calibration gradients, relative depth losses, scale-normalized pose supervision,
        and a training penalty for refinement that worsens supervised camera error.
        Confidence predicts relative-depth success (≤10% error); invalid pixels receive a separate bounded loss.
        G-LVP uses training-only standardized teacher targets and a contrastive term;
        validation retrieval, confidence reliability and calibration diagnostics are exported.
        These are experimental changes. They do not establish improved reconstruction accuracy.
        Selection remains the declared validation depth/pose composite; test data never choose checkpoints.

        ## 1. Install pinned extras

In [ ]:
# Keep Kaggle's matching Torch / torchvision / numerical stack.
import importlib.metadata as metadata
import os, sys, subprocess
from pathlib import Path
assert sys.version_info >= (3, 11), 'Use a current Kaggle Python image'
protected = ['torch','torchvision','numpy','scipy','pandas','Pillow','opencv-python-headless']
pins = []
for package in protected:
    try: pins.append(f'{package}=={metadata.version(package)}')
    except metadata.PackageNotFoundError: pass
constraint = Path('/kaggle/working/v46_stack_constraints.txt')
constraint.parent.mkdir(parents=True, exist_ok=True)
constraint.write_text('\n'.join(pins)+'\n')
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
    '-c',str(constraint),'timm==1.0.30','gsplat==1.5.3','ninja==1.13.2','einops==0.8.1'],check=True)
os.environ['OMP_NUM_THREADS']='1'
os.environ['MKL_NUM_THREADS']='1'
os.environ['MAX_JOBS']='2'
os.environ['NCCL_P2P_DISABLE']='1'
os.environ['NCCL_IB_DISABLE']='1'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
print('Pinned extras installed; Kaggle base packages constrained.')


## 2. Materialize the embedded source

In [ ]:
# Embedded source bundle: identical to the reviewable kaggle_v46/*.py files.
import base64, hashlib, io, json, zipfile
from pathlib import Path
import sys, os
BUNDLE_SHA256 = '79df7d019d7fc68cec1b0661cd962a3a5445acfa3838c02151db396d9d61c398'
BUNDLE_BASE64 = '...'

blob=base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(blob).hexdigest()==BUNDLE_SHA256
SOURCE_ROOT=Path('/kaggle/working/vjepa_v46_src')/BUNDLE_SHA256[:16]
SOURCE_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(SOURCE_ROOT)
for module in list(sys.modules):
    if module=='kaggle_v46' or module.startswith('kaggle_v46.'):
        del sys.modules[module]
sys.path.insert(0,str(SOURCE_ROOT))
os.environ['PYTHONPATH']=str(SOURCE_ROOT)+os.pathsep+os.environ.get('PYTHONPATH','')
print('Reviewable source extracted to',SOURCE_ROOT)


## 3. Choose data paths, run mode and experiment

        Main: `STUDY='main', VARIANT='full'`, 5/10/5 epochs.
        Matched G-LVP control: `STUDY='paired'`, `VARIANT='full'` or `'no_glvp'`;
        each job keeps the full 20-epoch schedule and five-view evaluation. Run separately.
        A contribution claim requires both completed jobs; no control result is assumed.
        G-LVP: `STUDY='glvp'`, 3/6/3 epochs, three-view train/eval; variants
        `A0_no_glvp`, `A1_glvp_0025`, `A2_no_conditioning`, `A3_glvp_0010`, `A4_glvp_0050`.
        Components: `STUDY='components'`, 2/4/0 epochs; `full`, `frozen_backbone`,
        `random_frozen_features`, `no_sim3`, `no_pointmap`, `no_camera`, `no_pairwise`,
        `no_reprojection`, `no_scale`, `no_render`, `no_epipolar`, `no_confidence`,
        `no_scene_tokens`, `no_refinement`, `no_descriptors`, `single_view`.
        Run each variant separately with the same seeds (e.g. 42, 43, 44). A random frozen feature control
        does not replace a matched alternative-pretraining study. Observed target images remain available
        to the model; G-LVP is not a withheld-view forecasting experiment.

        Resume requires the **same corrected source/config/data and both `last.pt` + `best.pt`**.
        For evaluation only, use the exact same study/variant/seed and an explicit corrected checkpoint.

In [ ]:
from kaggle_v46.config import default_config, apply_study

DATASET = 'co3d'
# No dataset attachments needed. Missing data downloads automatically into Kaggle temp.
DATA_ROOT = '/kaggle/temp/co3d_data' if DATASET=='co3d' else '/kaggle/temp/datasets/tum_rgbd'
ALLOW_DOWNLOAD = True
OUTPUT_ROOT = '/kaggle/working/vjepa_geo_v46'
ASSET_ROOT = '/kaggle/temp/vjepa_assets'
MODE = 'train'              # train | resume | evaluate
STUDY = 'main'              # main | paired | glvp | components (one job per variant/seed)
VARIANT = 'full'            # glvp example: A1_glvp_0025; components example: no_epipolar
SEED = 42
RUN_NAME = f'main_seed{SEED}' if STUDY=='main' else f'{STUDY}_{VARIANT}_seed{SEED}'
RESUME_CHECKPOINT = ''      # Attach a previous corrected output; path to last.pt.
                            # Its sibling best.pt is also required.
EVALUATE_CHECKPOINT = ''    # Optional corrected best.pt; blank uses this run's best.pt.
COMPANION_RUN = ''          # Completed OTHER dataset's v46 main-run directory, for joint curves.
SESSION_HOURS = 10.5        # Stops at a saved epoch boundary; resume to finish the full schedule.

cfg=default_config(DATASET)
cfg.update(data_root=DATA_ROOT,output_root=OUTPUT_ROOT,asset_root=ASSET_ROOT,
    mode=MODE,study=STUDY,variant=VARIANT,seed=SEED,run_name=RUN_NAME,
    resume_checkpoint=RESUME_CHECKPOINT,evaluate_checkpoint=EVALUATE_CHECKPOINT,
    companion_run=COMPANION_RUN,max_hours=SESSION_HOURS,allow_download=ALLOW_DOWNLOAD)
# Full-resolution corrected main experiment. Studies apply their explicit schedules below.
cfg.update(image_size=512,epochs=[5,10,5],eval_views=5,
    micro_batch_per_gpu=1,global_effective_batch=4,workers_per_gpu=1,
    lr_heads=1.5e-4,lr_backbone=2e-6,weight_decay=1e-4,
    amp=True,amp_dtype='float16',eval_amp=False,
    depth_bins=24,depth_chunk=4,checkpoint_activations=True,
    use_render=True,export_points=True,require_two_gpus=True)
cfg=apply_study(cfg)
CONFIG_PATH=Path('/kaggle/working/v46_config.json')
CONFIG_PATH.write_text(json.dumps(cfg,indent=2))
RUN_DIR=Path(OUTPUT_ROOT)/DATASET/STUDY/RUN_NAME
print(json.dumps({k:cfg[k] for k in ['dataset','study','variant','epochs','stage_k',
    'stage_k_probabilities','eval_views','global_effective_batch','lr_heads','lr_backbone',
    'image_size','use_render','camera_grid','camera_dim','camera_match_weight',
    'selection_rule','selection_pose_weight','selection_rotation_weight','splits']},indent=2))
print('Two GPU ranks × microbatch 1 × accumulate 2 = effective global batch 4')
print('Output:',RUN_DIR)
Path('/kaggle/temp').mkdir(parents=True,exist_ok=True)
import shutil
print('Raw data:',DATA_ROOT,'; pretrained assets:',ASSET_ROOT)
print('Free Kaggle temp storage (GiB):',round(shutil.disk_usage('/kaggle/temp').free/2**30,1))
print('Existing complete downloads are reused; missing data downloads automatically. No data attachment required.')


## 4. Verify both GPUs and actual differentiable CUDA rendering

In [ ]:
# Run CUDA compilation/rendering in a child process so the notebook holds no GPU model.
import subprocess, sys, os
from kaggle_v46.preflight import find_cuda_toolkit
os.environ['CUDA_HOME']=find_cuda_toolkit()
os.environ['PATH']=os.environ['CUDA_HOME']+'/bin:'+os.environ.get('PATH','')
subprocess.run([sys.executable,'-m','kaggle_v46.preflight'],check=True)


## 5. Train and evaluate with two GPU processes

In [ ]:
# One process per GPU; training, validation, selected-checkpoint test and exports run together.
command=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',
    '-m','kaggle_v46.train','--config',str(CONFIG_PATH)]
log_path=Path('/kaggle/working')/f'{DATASET}_{RUN_NAME}.log'
print('Launching:', ' '.join(command),flush=True)
with log_path.open('w') as logfile:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,bufsize=1,env=dict(os.environ,PYTHONUNBUFFERED='1'))
    try:
        for line in process.stdout:
            print(line,end='',flush=True);logfile.write(line);logfile.flush()
        returncode=process.wait()
    except BaseException:
        process.terminate();process.wait();raise
if returncode:
    raise RuntimeError(f'torchrun failed ({returncode}); see {log_path}. No successful run is claimed.')
state=json.loads((RUN_DIR/'run_state.json').read_text())
print(json.dumps(state,indent=2))
if state['status']!='complete':
    print('Saved an incomplete run. Attach its output, set MODE=resume and RESUME_CHECKPOINT=last.pt, then rerun.')


## 6. Download outputs

        `paper_artifacts.zip` contains figures, tables, configuration and provenance.
        Each cloud is exported separately under `window_clouds/`; full checkpoints are separate.
        The seven existing CO3D figures are retained, with a new confidence/G-LVP diagnostic.
        Historical TUM notebooks and results remain separate study evidence; this release asks for CO3D only.
        **Changed captions:** the CO3D pooled cloud is now a single-window reference overlay;
        the two TUM GS appendix figures now show corrected final-model history and RGB initial/refined cameras.
        The cross-dataset confidence panel needs both new runs. Do not retain the old captions or numerical claims.

In [ ]:
from IPython.display import FileLink, display
if (RUN_DIR/'run_state.json').exists():
    print((RUN_DIR/'run_state.json').read_text())
for name in ['paper_artifacts.zip','best.pt','last.pt','resolved_config.json',
             'data_manifest.json','figure_manifest.json','training_history.csv']:
    path=RUN_DIR/name
    if path.exists():display(FileLink(str(path)))
if (RUN_DIR/'figure_manifest.json').exists():
    figures=json.loads((RUN_DIR/'figure_manifest.json').read_text())
    print('Cross-dataset confidence panel complete:',figures['cross_dataset_panel_complete'])
    print('Read figure_manifest.json before replacing any paper figure or caption.')
print('Per-window raw/reference/aligned point clouds:',RUN_DIR/'window_clouds')
print('Full checkpoints are separate from the small figures/tables ZIP. Save both for resume.')


## 7. Optional completed-study aggregation

In [ ]:
# Optional: run after completing multiple named variant/seed jobs under OUTPUT_ROOT.
COLLECT_COMPLETED_STUDIES=False
if COLLECT_COMPLETED_STUDIES:
    from kaggle_v46.reporting import collect_studies
    print(collect_studies(OUTPUT_ROOT,DATASET))
# To regenerate a joint CO3D/TUM confidence plot after BOTH runs, set COMPANION_RUN
# to the other completed run, MODE='evaluate', and EVALUATE_CHECKPOINT to this run's
# corrected best.pt, then run the notebook again. No historical CSVs are copied in.
